## 1. Загрузка исходного файла

Скачиваем архив, извлекаем XLS, читаем таблицу. 

In [1]:
# Скачиваем архив
import logging

from classifier.pipeline.download import download_dataset
from experiments.settings import ExperimentSettings

experiment_settings = ExperimentSettings()

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
archive_path = download_dataset()

INFO classifier.data.download: Downloaded dataset: /Users/mikhail_afanasiev/Dev/netology/netology-aiepro-diplom/runtime/data/raw/cardiotocography.zip


In [2]:
# Извлечь XLS и сохранить в raw
from zipfile import ZipFile

from classifier.pipeline import dataset_schema

workbook_path = experiment_settings.raw_dir / experiment_settings.workbook_filename
with ZipFile(archive_path) as archive:
    workbook_path.write_bytes(archive.read(experiment_settings.workbook_filename))

workbook_path

PosixPath('/Users/mikhail_afanasiev/Dev/netology/netology-aiepro-diplom/runtime/data/raw/CTG.xls')

In [3]:
# Прочитать таблицу и показать первые строки
import pandas as pd

df = pd.read_excel(workbook_path, sheet_name=experiment_settings.raw_sheet_name, header=0, engine="xlrd")
df.head()

,FileName,Date,SegFile,b,e,LBE,LB,AC,FM,UC,...,C,D,E,AD,DE,LD,FS,SUSP,CLASS,NSP
0,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Variab10.txt,1996-12-01,CTG0001.txt,240.0,357.0,120.0,120.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,9.0,2.0
2,Fmcs_1.txt,1996-05-03,CTG0002.txt,5.0,632.0,132.0,132.0,4.0,0.0,4.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,6.0,1.0
3,Fmcs_1.txt,1996-05-03,CTG0003.txt,177.0,779.0,133.0,133.0,2.0,0.0,5.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,6.0,1.0
4,Fmcs_1.txt,1996-05-03,CTG0004.txt,411.0,1192.0,134.0,134.0,2.0,0.0,6.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,6.0,1.0


## 2. Обзор данных

- Лист `Description` - описание данных. Указаны 2126 измерений и классификаций сигналов сердечного ритма плода.
- Лист `Raw Data` — исходная **таблица измерений**. 
- Лист `Data` содержит вариант с временными признаками на единицу времени; готовые преобразования оттуда не используются (как будто их нет). Подобные преобразования можно повторить при необходимости

### Служебные поля и границы сегмента

| Колонка | Понятное описание | Роль / связь |
|---|---|---|
| `FileName` | Имя файла исходного КТГ-исследования | Возможный ключ группы; не доказанный ID пациента |
| `Date` | Дата исследования | Служебные сведения |
| `SegFile` | Имя файла сегмента | В `Description` отдельно не расшифрован; назначение следует из имени и значений |
| `b` | Момент начала сегмента | Вместе с `e` задаёт его границы; единицы в `Description` не указаны |
| `e` | Момент конца сегмента | `e - b` — длительность в единицах исходной шкалы |

### Входные измерения — кандидаты в признаки

Окончательный состав входов определим после EDA. Включение `LBE` и `DR` пока не согласовано; здесь перечислены исходные измерения, а не готовый набор для обучения.

| Колонка | Понятное описание | Роль / связь |
|---|---|---|
| `LBE` | Базовый уровень сердечного ритма, определённый медицинским экспертом | Экспертное измерение; его доступность в будущем сервисе нужно обсудить |
| `LB` | Базовый уровень сердечного ритма, рассчитанный SisPorto | Автоматический аналог `LBE` |
| `AC` | Количество ускорений сердечного ритма | Временные события; в `Data` есть показатель на единицу времени |
| `FM` | Количество движений плода | Временные события |
| `UC` | Количество сокращений матки | Временные события |
| `ASTV` | Процент времени с аномальной кратковременной вариабельностью ритма | Доля времени, % |
| `MSTV` | Средняя кратковременная вариабельность — быстрые изменения ритма | В `Description` написано `mSTV` |
| `ALTV` | Процент времени с аномальной долговременной вариабельностью ритма | Доля времени, % |
| `MLTV` | Средняя долговременная вариабельность — более медленные изменения ритма | В `Description` написано `mLTV` |
| `DL` | Количество лёгких замедлений сердечного ритма | Временные события |
| `DS` | Количество выраженных замедлений сердечного ритма | Временные события |
| `DP` | Количество продолжительных замедлений сердечного ритма | Временные события |
| `DR` | Количество повторяющихся замедлений сердечного ритма | Авторы исключили из набора признаков `Data`, указав `p(K-W)=1`; мы сначала исследуем сами |
| `Width` | Ширина гистограммы значений сердечного ритма | Диапазон распределения; проверим связь с `Max - Min` |
| `Min` | Нижняя граница гистограммы сердечного ритма | Минимальная частота ритма в гистограмме |
| `Max` | Верхняя граница гистограммы сердечного ритма | Максимальная частота ритма в гистограмме |
| `Nmax` | Число пиков гистограммы | Сколько выраженных максимумов у распределения |
| `Nzeros` | Число нулей гистограммы | Не означает число остановок сердца; точное определение нулей в листе не приведено |
| `Mode` | Мода гистограммы | Наиболее часто встречающееся значение ритма |
| `Mean` | Среднее по гистограмме | Средний уровень ритма |
| `Median` | Медиана гистограммы | Центральное значение распределения |
| `Variance` | Дисперсия гистограммы | Характеристика разброса значений |
| `Tendency` | Направление асимметрии гистограммы | По Description: −1 — влево, 0 — симметрично, 1 — вправо |

### Индикаторы паттернов

Эти колонки описывают разметку паттерна и не входят во входные признаки.

| Колонка | Понятное описание | Роль / связь |
|---|---|---|
| `A` | Паттерн спокойного сна | Индикатор категории CLASS, не входной признак |
| `B` | Паттерн REM-сна | Индикатор категории CLASS, не входной признак |
| `C` | Паттерн спокойного бодрствования | Индикатор категории CLASS, не входной признак |
| `D` | Паттерн активного бодрствования | Индикатор категории CLASS, не входной признак |
| `E` | Пятый паттерн; в Description вместо `E` указан `SH`: паттерн со сдвигами (`A` или подозрительный со сдвигами) | Несовпадение обозначений в источнике; связь с CLASS=5 проверим по данным |
| `AD` | Паттерн ускорений и замедлений; авторы связывают со стрессовой ситуацией | Индикатор категории CLASS |
| `DE` | Паттерн замедлений; авторы связывают со стимуляцией блуждающего нерва | Индикатор категории CLASS |
| `LD` | Паттерн с преобладанием замедлений | Индикатор категории CLASS |
| `FS` | Плоский/синусоидальный паттерн; в описании указан как патологическое состояние | Индикатор категории CLASS |
| `SUSP` | Подозрительный паттерн | Индикатор категории CLASS; не путать с NSP=2 |

### Две целевые колонки

| Колонка | Понятное описание | Роль / связь |
|---|---|---|
| `CLASS` | Код одного из десяти паттернов КТГ | Цель: по порядку описания A, B, C, D, SH/E, AD, DE, LD, FS, SUSP → 1…10 |
| `NSP` | Итоговая экспертная категория состояния | Вторая цель: 1 — нормальная, 2 — подозрительная, 3 — патологическая |

**Как связаны параметры:** измерения описывают сегмент; `A…SUSP` и `CLASS` представляют разметку его паттерна. `NSP` — другая целевая оценка. В `Description` нет таблицы, доказывающей однозначный перевод CLASS → NSP: эту гипотезу проверим отдельно. Индикаторы паттернов нельзя подавать в модель вместе с измерениями — это раскрывает целевую разметку.


## 3. Пропуски, дубли и корректность значений

In [5]:
print(f"Размер исходной таблицы: \n\t{df.shape[0]} строк\n\t{df.shape[1]} колонок")

Размер исходной таблицы: 
	2130 строк
	40 колонок


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2130 entries, 0 to 2129
Data columns (total 40 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   FileName  2126 non-null   str           
 1   Date      2126 non-null   datetime64[us]
 2   SegFile   2126 non-null   str           
 3   b         2126 non-null   float64       
 4   e         2126 non-null   float64       
 5   LBE       2126 non-null   float64       
 6   LB        2126 non-null   float64       
 7   AC        2126 non-null   float64       
 8   FM        2127 non-null   float64       
 9   UC        2127 non-null   float64       
 10  ASTV      2127 non-null   float64       
 11  MSTV      2127 non-null   float64       
 12  ALTV      2127 non-null   float64       
 13  MLTV      2127 non-null   float64       
 14  DL        2128 non-null   float64       
 15  DS        2128 non-null   float64       
 16  DP        2128 non-null   float64       
 17  DR        2128 non-null  

Количество non-null значений во всех колонках отличается от количества строк - значит есть пропуски.

In [7]:
rows_with_nan = df[df.isna().any(axis=1)]
with pd.option_context("display.max_columns", None):
    display(rows_with_nan)

,FileName,Date,SegFile,b,e,LBE,LB,AC,FM,UC,ASTV,MSTV,ALTV,MLTV,DL,DS,DP,DR,Width,Min,Max,Nmax,Nzeros,Mode,Mean,Median,Variance,Tendency,A,B,C,D,E,AD,DE,LD,FS,SUSP,CLASS,NSP
0,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2127,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2128,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2129,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,564.0,23.0,87.0,7.0,91.0,50.7,16.0,1.0,4.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Строки 0 и 2127 - полностью пустые - в исходном xls-файле действительно есть пустые строки между шапкой таблицы и в конце  
Строки 2128 и 2129 заполнены частично, но судя исходному файлу это ячейки с вычисляемыми по формулам значениями  
Эти 4 строки нужно исключить из датасета

In [8]:
df = df.drop(rows_with_nan.index).reset_index(drop=True)
print(f"Размер обновленной таблицы: \n\t{df.shape[0]} строк\n\t{df.shape[1]} колонок")

Размер обновленной таблицы: 
	2126 строк
	40 колонок


In [9]:
identifier_cols = ["FileName", "SegFile"]
date_cols = ["Date"]
timestamp_cols = ["b", "e"]
target_cols = ["CLASS", "NSP"]
pattern_cols = ["A", "B", "C", "D", "E", "AD", "DE", "LD", "FS", "SUSP"]
categorical_cols = ["CLASS", "NSP", "Tendency"]

numeric_cols = (
    df.drop(columns=[*categorical_cols, *timestamp_cols, *pattern_cols]).select_dtypes(include=["float64"]).columns
)
feature_cols = df.columns.drop([*identifier_cols, *date_cols, *timestamp_cols, *pattern_cols, *target_cols]).tolist()

In [10]:
for one_col in [*identifier_cols, *date_cols]:
    print(f"------------------------\n{one_col}")
    unique_values = df[one_col].unique()
    unique_values_count = len(unique_values)
    print(f"Уникальных значений: {unique_values_count}")

------------------------
FileName
Уникальных значений: 352
------------------------
SegFile
Уникальных значений: 2126
------------------------
Date
Уникальных значений: 48


In [11]:
print(f"Повторяющихся идентификаторов сегментов: {df['SegFile'].duplicated().sum()}")

print(f"Полностью одинаковых строк: {df.duplicated().sum()}")

print(f"Совпадения всех полей, кроме SegFile: {df.drop(columns='SegFile').duplicated().sum()}")

Повторяющихся идентификаторов сегментов: 0
Полностью одинаковых строк: 0
Совпадения всех полей, кроме SegFile: 10


In [12]:
duplicates = df[df.duplicated(subset=df.columns.drop("SegFile"), keep=False)]

with pd.option_context("display.max_columns", None):
    display(duplicates.sort_values(by=df.columns.drop("SegFile").tolist()))

,FileName,Date,SegFile,b,e,LBE,LB,AC,FM,UC,ASTV,MSTV,ALTV,MLTV,DL,DS,DP,DR,Width,Min,Max,Nmax,Nzeros,Mode,Mean,Median,Variance,Tendency,A,B,C,D,E,AD,DE,LD,FS,SUSP,CLASS,NSP
229,Aast_10.txt,1996-05-06,CTG0230.txt,0.0,1199.0,123.0,123.0,3.0,4.0,0.0,52.0,0.8,2.0,15.4,0.0,0.0,0.0,0.0,90.0,50.0,140.0,7.0,0.0,129.0,128.0,130.0,4.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,1.0
787,Aast_10.txt,1996-05-06,CTG0788.txt,0.0,1199.0,123.0,123.0,3.0,4.0,0.0,52.0,0.8,2.0,15.4,0.0,0.0,0.0,0.0,90.0,50.0,140.0,7.0,0.0,129.0,128.0,130.0,4.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,1.0
790,Aast_10.txt,1996-05-06,CTG0791.txt,176.0,928.0,123.0,123.0,2.0,3.0,0.0,50.0,0.9,4.0,14.8,0.0,0.0,0.0,0.0,82.0,58.0,140.0,7.0,0.0,129.0,128.0,130.0,5.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,1.0
791,Aast_10.txt,1996-05-06,CTG0792.txt,176.0,928.0,123.0,123.0,2.0,3.0,0.0,50.0,0.9,4.0,14.8,0.0,0.0,0.0,0.0,82.0,58.0,140.0,7.0,0.0,129.0,128.0,130.0,5.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,1.0
233,Aast_10.txt,1996-05-06,CTG0234.txt,276.0,596.0,123.0,123.0,0.0,0.0,0.0,49.0,0.8,7.0,13.8,0.0,0.0,0.0,0.0,74.0,63.0,137.0,2.0,0.0,129.0,127.0,129.0,2.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
234,Aast_10.txt,1996-05-06,CTG0235.txt,276.0,596.0,123.0,123.0,0.0,0.0,0.0,49.0,0.8,7.0,13.8,0.0,0.0,0.0,0.0,74.0,63.0,137.0,2.0,0.0,129.0,127.0,129.0,2.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
323,Mabp_4.txt,1996-05-02,CTG0324.txt,179.0,666.0,135.0,135.0,0.0,0.0,0.0,62.0,0.5,71.0,6.9,0.0,0.0,0.0,0.0,97.0,71.0,168.0,3.0,0.0,143.0,142.0,144.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,9.0,3.0
324,Mabp_4.txt,1996-05-02,CTG0325.txt,179.0,666.0,135.0,135.0,0.0,0.0,0.0,62.0,0.5,71.0,6.9,0.0,0.0,0.0,0.0,97.0,71.0,168.0,3.0,0.0,143.0,142.0,144.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,9.0,3.0
326,Mabp_6.txt,1996-05-02,CTG0327.txt,0.0,809.0,144.0,144.0,0.0,15.0,0.0,76.0,0.4,61.0,10.6,0.0,0.0,0.0,0.0,81.0,71.0,152.0,3.0,0.0,145.0,144.0,146.0,2.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,10.0,2.0
333,Mabp_6.txt,1996-05-02,CTG0334.txt,0.0,809.0,144.0,144.0,0.0,15.0,0.0,76.0,0.4,61.0,10.6,0.0,0.0,0.0,0.0,81.0,71.0,152.0,3.0,0.0,145.0,144.0,146.0,2.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,10.0,2.0


- Столбец `SegFile` является уникальным идентификатором.
- Полностью идентичных строк нет
- Есть 8 групп по 2 повторяющихся результата и одзна группа из трех повторяющихся результатов в рамках одного исходного файла, но разных сегментов. Поскольку совпадает не только название файла и дата, но также, несмотря на разный сегмент - временные метки `b` и `e` и результаты измерений - с высокой вероятностью это задублировавшиеся сегменты. Дубли лучше удалить из датасета, тем более их всего около 0.5%. 

In [13]:
df = df.drop_duplicates(subset=df.columns.drop("SegFile"), keep="first").reset_index(drop=True)
print(f"Размер обновленной таблицы: \n\t{df.shape[0]} строк\n\t{df.shape[1]} колонок")

Размер обновленной таблицы: 
	2116 строк
	40 колонок


In [14]:
conflicts = df.groupby(feature_cols, dropna=False)[target_cols].transform("nunique").gt(1).any(axis=1)
with pd.option_context("display.max_columns", None):
    display(df.loc[conflicts].sort_values(feature_cols))

,FileName,Date,SegFile,b,e,LBE,LB,AC,FM,UC,ASTV,MSTV,ALTV,MLTV,DL,DS,DP,DR,Width,Min,Max,Nmax,Nzeros,Mode,Mean,Median,Variance,Tendency,A,B,C,D,E,AD,DE,LD,FS,SUSP,CLASS,NSP
1371,S0001085.dat,1996-10-03,CTG1383.txt,1925.0,2500.0,121.0,121.0,0.0,0.0,2.0,24.0,1.5,0.0,0.0,3.0,0.0,1.0,0.0,68.0,71.0,139.0,2.0,0.0,123.0,106.0,108.0,53.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,7.0,2.0
1372,S0001085.dat,1996-10-03,CTG1384.txt,1925.0,2500.0,121.0,121.0,0.0,0.0,2.0,24.0,1.5,0.0,0.0,3.0,0.0,1.0,0.0,68.0,71.0,139.0,2.0,0.0,123.0,106.0,108.0,53.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,7.0,1.0
1105,S0001041.dat,1996-08-28,CTG1113.txt,569.0,1320.0,122.0,122.0,0.0,0.0,0.0,19.0,1.9,0.0,15.1,0.0,0.0,0.0,0.0,39.0,103.0,142.0,1.0,0.0,120.0,120.0,122.0,3.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
1106,S0001041.dat,1996-08-28,CTG1114.txt,569.0,1320.0,122.0,122.0,0.0,0.0,0.0,19.0,1.9,0.0,15.1,0.0,0.0,0.0,0.0,39.0,103.0,142.0,1.0,0.0,120.0,120.0,122.0,3.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,1.0
841,S0001015.dat,1996-07-18,CTG0849.txt,2094.0,3176.0,138.0,138.0,2.0,0.0,4.0,41.0,0.8,8.0,10.3,0.0,0.0,0.0,0.0,51.0,105.0,156.0,4.0,0.0,142.0,142.0,143.0,2.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
842,S0001015.dat,1996-07-18,CTG0850.txt,2087.0,3163.0,138.0,138.0,2.0,0.0,4.0,41.0,0.8,8.0,10.3,0.0,0.0,0.0,0.0,51.0,105.0,156.0,4.0,0.0,142.0,142.0,143.0,2.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,1.0
799,S0001011.dat,1996-07-16,CTG0807.txt,70.0,928.0,146.0,146.0,0.0,0.0,1.0,53.0,0.5,32.0,8.2,0.0,0.0,0.0,0.0,27.0,136.0,163.0,2.0,1.0,160.0,156.0,158.0,2.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
800,S0001011.dat,1996-07-16,CTG0808.txt,70.0,928.0,146.0,146.0,0.0,0.0,1.0,53.0,0.5,32.0,8.2,0.0,0.0,0.0,0.0,27.0,136.0,163.0,2.0,1.0,160.0,156.0,158.0,2.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,5.0,2.0


- В датасете обнаружена конфликтующая разметка
- Для CLASS — 3 конфликтующие пары
- Для NSP — 2 конфликтующие пары
- Одна пара конфликтует по обеим переменным
- Среди конфликтов присутствуют записи с идентичными временными метками и с отличающимися

Поскольку CLASS и NSP — две независимые целевые переменные - конфликт правильно решать отдельно для каждой. Поскольку эксперты оценивали не только числовые данные, но и вполне вероятно - полную картину, то конфликт в данном случае не означает, что одна из записей в паре неверная. Поскольку доля конфликтующих записей в датасете мала (около 0.4%), для того чтобы избавиться от неоднозначных примеров, в данном случае можно исключить эти записи из выборки.

In [15]:
df = df.loc[~conflicts].reset_index(drop=True)
print(f"Размер обновленной таблицы: \n\t{df.shape[0]} строк\n\t{df.shape[1]} колонок")

Размер обновленной таблицы: 
	2108 строк
	40 колонок


In [16]:
# Границы сегментов
assert (df["e"] > df["b"]).all(), "Есть сегменты с e <= b"

# Отсутствие пропусков
assert not df.isna().any().any(), "Есть пропущенные значения"

# Уникальность сегментов
assert df["SegFile"].is_unique, "Есть повторяющиеся SegFile"

# Допустимые значения целевых переменных
assert df["CLASS"].isin(dataset_schema.ClassificationTarget.CLASS.mappings).all(), "Некорректные CLASS"
assert df["NSP"].isin(dataset_schema.ClassificationTarget.NSP.mappings).all(), "Некорректные NSP"

# Соответствие CLASS и one-hot столбцов
assert (
    df[pattern_cols].isin([0, 1]).all(axis=1)  # Все значения только 0 или 1
    & df[pattern_cols].sum(axis=1).eq(1)  # Ровно одна 1 в строке
    & df[pattern_cols]
    .idxmax(axis=1)
    .eq(df["CLASS"].map(dataset_schema.ClassificationTarget.CLASS.mappings))  # Соответствие CLASS и столбца с 1
).all(), "Некорректная разметка CLASS"

# Отсутствие полных дубликатов без SegFile
assert not df.duplicated(subset=df.columns.drop("SegFile")).any(), "Есть дубликаты сегментов"


# Группы с одинаковыми входными признаками, но разными CLASS или NSP
conflicts = df.groupby(feature_cols, dropna=False)[target_cols].transform("nunique").gt(1).any(axis=1)
assert not conflicts.any(), "Есть строки с противоречивой разметкой"


print("Валидация прошла успешно!")

Валидация прошла успешно!


In [17]:
for one_col in categorical_cols:
    print(f"------------------------\n{one_col}")
    unique_values = df[one_col].unique()
    unique_values_count = len(unique_values)
    print(f"Уникальных значений: {unique_values_count}")
    if len(unique_values) < 20:
        print(f"Уникальные значения: {df[one_col].sort_values().unique()}")

------------------------
CLASS
Уникальных значений: 10
Уникальные значения: [ 1.  2.  3.  4.  5.  6.  7.  8.  9. 10.]
------------------------
NSP
Уникальных значений: 3
Уникальные значения: [1. 2. 3.]
------------------------
Tendency
Уникальных значений: 3
Уникальные значения: [-1.  0.  1.]


В категориальных столбцах датасета все значения находятся в ожидаемом диапазоне:
- Tendency: -1, 0, 1
- NSP: 1, 2, 3
- CLASS: от 1 до 10
- NSP и CLASS - это target-столбцы, поэтому не имеет значения оставить в них числа или строки. Строки проще интерпретировать, но в случае использования нейросетевых моделей потребуется их преобразовать в числа. Можно оставить числа, а для интерпретации результата пользоваться маппингом.
- При этом для `NSP`: Normal < Suspect < Pathologic и числовое представление тут имеет дополнительный смысл, отражающий близость значений, тогда как для CLASS это не так.
- `Tendency` правильно оставить в числовом представлении, т.к. числа отражают удаленность категорий друг от друга

In [18]:
df_categorical = df[categorical_cols].dropna()

categorical_report = pd.DataFrame(index=categorical_cols)

categorical_report["na_cnt"] = df_categorical.isna().sum() * 100
categorical_report["na_%"] = df_categorical.isna().mean() * 100
categorical_report["unique"] = df_categorical.nunique(dropna=True)
categorical_report["mode"] = df_categorical.mode().iloc[0]
categorical_report["mode_freq_%"] = df_categorical.apply(lambda x: x.value_counts(normalize=True).iloc[0] * 100).round(
    2,
)
categorical_report["cardinality_%"] = (100 * df_categorical.nunique(dropna=True) / len(df_categorical)).round(2)
categorical_report["high_cardinality"] = categorical_report["cardinality_%"] > 10
categorical_report["quasi_constant"] = categorical_report["mode_freq_%"] > 90

with pd.option_context("display.max_columns", None):
    display(categorical_report.T)

,CLASS,NSP,Tendency
na_cnt,0,0,0
na_%,0.0,0.0,0.0
unique,10,3,3
mode,2.0,1.0,0.0
mode_freq_%,27.23,77.89,52.51
cardinality_%,0.47,0.14,0.14
high_cardinality,False,False,False
quasi_constant,False,False,False


- После очистки в CLASS осталось 10 категорий, в NSP и Tendency — по 3.
- Пропусков нет. 
- Самый частый паттерн (CLASS) - 2, он встречается в 27,23% записей. 
- Для Tendency чаще всего встречается значение 0 — симметричная гистограмма (52,51% записей).
- Tendency является входным признаком, а CLASS и NSP — целевыми колонками.
- В NSP заметен дисбаланс: нормальное состояние составляет 77,89% записей. Поэтому в модели важно будет оценивать распознавание подозрительных и патологических случаев. 
- Ни в одной из трёх колонок доля самой частой категории не превышает принятый в отчёте порог 90%.

In [19]:
df_numeric = df[numeric_cols]

numeric_report = df_numeric.describe(percentiles=[0.1, 0.25, 0.75, 0.9]).T
numeric_report["var"] = numeric_report["std"] ** 2
numeric_report["iqr"] = numeric_report["75%"] - numeric_report["25%"]
lower = numeric_report["25%"] - 1.5 * numeric_report["iqr"]
upper = numeric_report["75%"] + 1.5 * numeric_report["iqr"]
numeric_report["lower"] = lower
numeric_report["upper"] = upper
is_outlier = (df_numeric.lt(lower)) | (df_numeric.gt(upper))
numeric_report["na cnt"] = df_numeric.isna().sum()
numeric_report["na_%"] = df_numeric.isna().mean() * 100
numeric_report["outliers cnt"] = is_outlier.sum()
numeric_report["outliers %"] = is_outlier.mean() * 100
numeric_report["skew"] = df_numeric.skew()
numeric_report = numeric_report.T

with pd.option_context("display.max_columns", None):
    display(numeric_report.round(3))

,LBE,LB,AC,FM,UC,ASTV,MSTV,ALTV,MLTV,DL,DS,DP,DR,Width,Min,Max,Nmax,Nzeros,Mode,Mean,Median,Variance
count,2108.000,2108.000,2108.000,2108.000,2108.000,2108.000,2108.000,2108.000,2108.000,2108.000,2108.000,2108.000,2108.0,2108.000,2108.000,2108.000,2108.000,2108.000,2108.000,2108.000,2108.000,2108.000
mean,133.313,133.313,2.738,7.293,3.681,47.040,1.335,9.806,8.169,1.581,0.003,0.126,0.0,70.580,93.553,164.132,4.081,0.325,137.457,134.615,138.109,18.898
std,9.834,9.834,3.570,37.278,2.847,17.184,0.885,18.358,5.632,2.506,0.058,0.465,0.0,39.037,29.569,17.945,2.954,0.708,16.397,15.588,14.450,29.048
min,106.000,106.000,0.000,0.000,0.000,12.000,0.200,0.000,0.000,0.000,0.000,0.000,0.0,3.000,50.000,122.000,0.000,0.000,60.000,73.000,77.000,0.000
10%,121.000,121.000,0.000,0.000,0.000,23.700,0.400,0.000,1.300,0.000,0.000,0.000,0.0,22.000,54.000,143.000,1.000,0.000,120.000,115.000,120.000,1.000
25%,126.000,126.000,0.000,0.000,1.000,32.000,0.700,0.000,4.600,0.000,0.000,0.000,0.0,37.000,67.000,152.000,2.000,0.000,129.000,125.000,129.000,2.000
75%,140.000,140.000,4.000,2.000,5.000,61.000,1.700,11.000,10.800,3.000,0.000,0.000,0.0,100.250,120.000,174.000,6.000,0.000,148.000,145.000,148.000,24.000
90%,146.000,146.000,8.000,9.000,7.000,68.000,2.500,38.000,14.900,5.000,0.000,0.000,0.0,126.000,133.000,189.000,8.000,1.000,156.000,153.000,155.000,53.000
max,160.000,160.000,26.000,564.000,23.000,87.000,7.000,91.000,50.700,16.000,1.000,4.000,0.0,180.000,159.000,238.000,18.000,10.000,187.000,182.000,186.000,269.000
var,96.702,96.702,12.742,1389.645,8.104,295.276,0.784,337.028,31.719,6.279,0.003,0.217,0.0,1523.916,874.301,322.025,8.725,0.502,268.864,242.976,208.805,843.790


В проверяемых числовых колонках пропусков нет. Значения имеют разный масштаб: например, MSTV лежит в диапазоне 0,2–7, а FM — 0–564. Для линейной модели и нейросети потребуется масштабирование признаков.

- DR во всех строках равен нулю и не помогает различать записи.
- У LBE и LB совпадают сводные показатели. Чтобы решить, дублируют ли они друг друга, нужно отдельно сравнить значения построчно.
- У FM, ALTV и Variance есть длинный хвост больших значений. По правилу IQR необычными отмечены около 14,66%, 14,42% и 8,73% значений соответственно. Это повод изучить распределения, а не автоматически удалить строки.
- Для редких счётчиков DS, DP и Nzeros межквартильный размах равен нулю, поэтому правило IQR отмечает любое ненулевое значение. Такие значения могут быть обычными редкими событиями.


### Выводы по первичному EDA

- После очистки осталось **2108 строк и 40 колонок**, пропусков нет. Удалены служебные строки, повторы и записи с конфликтующей разметкой.
- **FileName:** одному исследованию соответствуют несколько сегментов. Это нужно учитывать при разбиении данных.
- **NSP и CLASS:** классы несбалансированы. Normal занимает 77,89% записей, а самый редкий паттерн C представлен всего 50 записями. Качество нужно оценивать отдельно по классам.
- **DR:** во всех строках равен нулю и не помогает различать записи.
- **LBE и LB:** сводные показатели совпадают; нужно проверить, одинаковы ли значения построчно.
- **FM, ALTV и Variance:** встречаются редкие большие значения. Их нельзя автоматически считать ошибками и удалять.
- **DS, DP и Nzeros:** из-за большого количества нулей правило IQR отмечает любое ненулевое значение как выброс. Здесь оно требует осторожной интерпретации.
- **A–SUSP:** индикаторы целевой разметки, поэтому их нужно исключить из входов модели. Числовые измерения имеют разные масштабы и потребуют масштабирования для линейной модели и MLP.


## 4. Сохранение очищенной таблицы

Сохраняется результат очистки — 2108 строк со всеми колонками. Служебные поля нужны для разбиения, а индикаторы паттернов — для проверки разметки. Выбор входов модели выполняется отдельно в следующем notebook. Файл CSV сохраняется без индекса pandas; дата при чтении восстанавливается явно.

In [20]:
prepared_dir = experiment_settings.prepared_dir
prepared_dir.mkdir(parents=True, exist_ok=True)
cleaned_path = prepared_dir / "ctg_cleaned.csv"
df.to_csv(cleaned_path, index=False)
cleaned_path

PosixPath('/Users/mikhail_afanasiev/Dev/netology/netology-aiepro-diplom/runtime/data/prepared/ctg_cleaned.csv')